# Exercício Prático — Aula 07: Coleta via APIs REST

**Disciplina:** Análise e Ciência de Dados (Especialização, IFRN) — Unidade 2, Aula 07

Este exercício é **para praticar** os conceitos da aula (requisição GET, paginação, tratamento de erro, salvar em JSON/CSV) com dados reais do **IBGE** e do **TSE**. Ele não precisa ser entregue — é o aquecimento antes do **mini-projeto**, que é a avaliação desta aula (ver enunciado em separado no Moodle, prazo **20/10/2026**).

Preencha o código onde está escrito `# TODO`. Use o notebook `Aula07_Coleta_via_APIs_REST.ipynb` (o material desta aula) como referência — os padrões de código são os mesmos.

In [40]:
import requests
import json
import csv
import pandas as pd
from pathlib import Path

DADOS = Path("dados_exercicio")
DADOS.mkdir(exist_ok=True)

## Parte A — IBGE: municípios de outro estado

1. Escolha um estado **diferente** do Rio Grande do Norte (use a sigla de duas letras, por exemplo `"PB"`, `"CE"`, `"PE"`, `"BA"`...).
2. Monte a URL seguindo o mesmo padrão usado na aula para o RN:
   `https://servicodados.ibge.gov.br/api/v1/localidades/estados/<UF>/municipios`
3. Faça o GET, confira o `status_code` e descubra quantos municípios o estado escolhido tem.

In [161]:
UF_ESCOLHIDA = "RJ"  # TODO: troque por uma sigla de UF válida, diferente de RN
UF_ERRADA = "XX"
# TODO: monte a URL usando UF_ESCOLHIDA
url = f'https://servicodados.ibge.gov.br/api/v1/localidades/estados/{UF_ESCOLHIDA}/municipios'

print(url)
# TODO: faça requests.get(...) e guarde em uma variável `resposta`
resposta = requests.get(
    url,
    timeout=10,
)
dados = resposta.json()


# TODO: imprima resposta.status_code
print(resposta.status_code)

# TODO: converta para uma lista Python com resposta.json() e guarde em `municipios_uf`

with open(DADOS / 'municipios_uf.json',"w", encoding='utf-8') as arquivo:
    json.dump(dados,arquivo,indent=4,ensure_ascii=False)

# TODO: imprima quantos municípios foram encontrados
with open(DADOS / 'municipios_uf.json',"r", encoding='utf-8') as arquivo:
    payload = json.load(arquivo)
    
print(payload)

df = pd.DataFrame(payload)

df['nome'].count()

https://servicodados.ibge.gov.br/api/v1/localidades/estados/RJ/municipios
200
[{'id': 3300100, 'nome': 'Angra dos Reis', 'microrregiao': {'id': 33013, 'nome': 'Baía da Ilha Grande', 'mesorregiao': {'id': 3305, 'nome': 'Sul Fluminense', 'UF': {'id': 33, 'sigla': 'RJ', 'nome': 'Rio de Janeiro', 'regiao': {'id': 3, 'sigla': 'SE', 'nome': 'Sudeste'}}}}, 'regiao-imediata': {'id': 330002, 'nome': 'Angra dos Reis', 'regiao-intermediaria': {'id': 3301, 'nome': 'Rio de Janeiro', 'UF': {'id': 33, 'sigla': 'RJ', 'nome': 'Rio de Janeiro', 'regiao': {'id': 3, 'sigla': 'SE', 'nome': 'Sudeste'}}}}}, {'id': 3300159, 'nome': 'Aperibé', 'microrregiao': {'id': 33002, 'nome': 'Santo Antônio de Pádua', 'mesorregiao': {'id': 3301, 'nome': 'Noroeste Fluminense', 'UF': {'id': 33, 'sigla': 'RJ', 'nome': 'Rio de Janeiro', 'regiao': {'id': 3, 'sigla': 'SE', 'nome': 'Sudeste'}}}}, 'regiao-imediata': {'id': 330012, 'nome': 'Santo Antônio de Pádua', 'regiao-intermediaria': {'id': 3304, 'nome': 'Campos dos Goytacaze

92

In [64]:
# TODO: compare com o RN (que tem 167 municípios).
# O estado escolhido tem mais ou menos municípios que o RN? Qual a diferença (em número)?
total_municipios = df['nome'].count()

verificar = total_municipios >= 167
verificar
if verificar == False:
    print(f'diferença entre municipios {167 - total_municipios}')
else:
    print(f'diferença entre municipios{total_municipios - 167}')



diferença entre municipios 75


## Parte B — TSE: paginação com um termo à sua escolha

1. Escolha um termo de busca relacionado a eleições que **não** seja `"eleições"` (ex.: `"biometria"`, `"urna eletrônica"`, `"propaganda eleitoral"`, `"prestação de contas"`, `"candidatos"`...).
2. Implemente um laço que pagine pelo endpoint `package_search` (igual fizemos na aula) usando `rows`/`start`, **até coletar todos os resultados** — não só a primeira página.
3. Ao final, imprima quantos conjuntos de dados foram encontrados no total e liste os nomes (`name`) de todos eles.

In [102]:
BASE = "https://dadosabertos.tse.jus.br/api/3/action/package_search"
TERMO = "eleitorado"  # TODO: pode trocar por outro termo de sua escolha

def buscar_pagina(termo, start, rows=5):
    # TODO: faça o GET em BASE com os parâmetros q, rows e start
    resposta =  requests.get(BASE, params={'q':termo,'start':start,'rows':rows},
                             timeout=10,)
   # TODO: retorne resposta.json()["result"]
    resposta.raise_for_status()
    return resposta.json()['result']
    

# TODO: escreva o laço de paginação (veja o padrão usado na Parte 2 da aula)
# Dica: pare quando start + rows >= count
todos = []
start = 0
rows = 5
while True:
    pagina = buscar_pagina("eleitorado", start, rows)
    todos.extend(pagina["results"])
    if start + rows >= pagina["count"]: # já coletamos tudo
        break
    start += rows

In [188]:
# TODO: imprima o total de conjuntos encontrados e os nomes de cada um
for i in todos:
 print(f"conjunto de dados : {i['name']}")

print(f"total encontrado: {len(todos)} ")


df_tse = pd.DataFrame(todos)

print(df_tse['name'].size)
df_tse.loc[0:20,['author','name']]


conjunto de dados : transferencia-do-eleitorado
conjunto de dados : rae-requerimento-de-alistamento-eleitoral
conjunto de dados : eleitorado-atual
conjunto de dados : eleitorado-2026
conjunto de dados : eleitorado-2024
conjunto de dados : eleitorado-2022
conjunto de dados : eleitorado-2020
conjunto de dados : eleitorado-2018
conjunto de dados : eleitorado-2016
conjunto de dados : eleitorado-2014
conjunto de dados : eleitorado-2012
conjunto de dados : eleitorado-2010
conjunto de dados : eleitorado-2008
conjunto de dados : eleitorado-2006
conjunto de dados : eleitorado-2004
conjunto de dados : eleitorado-2002
conjunto de dados : eleitorado-2000
conjunto de dados : eleitorado-1998
conjunto de dados : eleitorado-1996
conjunto de dados : eleitorado-1994
total encontrado: 20 
20


,author,name
0,,transferencia-do-eleitorado
1,,rae-requerimento-de-alistamento-eleitoral
2,,eleitorado-atual
3,,eleitorado-2026
4,,eleitorado-2024
5,,eleitorado-2022
6,,eleitorado-2020
7,,eleitorado-2018
8,,eleitorado-2016
9,,eleitorado-2014


## Parte C — Tratando um erro de propósito

1. Faça uma requisição que você **sabe** que vai dar erro — por exemplo, peça os municípios de uma UF inexistente (`"ZZ"`) na API do IBGE, ou peça um `package_show` com um `id` que não existe no TSE.
2. Trate o erro com `try`/`except` (use `response.raise_for_status()`), **sem deixar o programa quebrar** — em vez disso, imprima uma mensagem amigável.

In [168]:
# TODO: tente buscar um recurso que não existe (UF "ZZ" no IBGE, por exemplo)
# e trate o erro com try/except + raise_for_status(), imprimindo uma mensagem amigável em vez de travar
url_errada = f'https://servicodados.ibge.gov.bs/{UF_ERRADA}/municipios'


def buscar_com_seguranca(url, params=None, timeout=10):
    try:
        resposta = requests.get(url, params=params, timeout=timeout)
        resposta.raise_for_status() # transforma status de erro em exceção  
        return resposta.json()
    except requests.exceptions.Timeout:
        print("A API demorou demais para responder.")
    except requests.exceptions.HTTPError as e:
        print(f"Erro HTTP: {e}")
    except requests.exceptions.RequestException as e:
        print(f"Erro de rede/requisição: {e}")
    return None

buscar_com_seguranca(url_errada,params=None,timeout=10)

Erro de rede/requisição: HTTPSConnectionPool(host='servicodados.ibge.gov.bs', port=443): Max retries exceeded with url: /XX/municipios (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001C907972750>: Failed to resolve 'servicodados.ibge.gov.bs' ([Errno 11001] getaddrinfo failed)"))


## Parte D — Salvando tudo

Salve, em `dados_exercicio/`:
- a lista de municípios do estado escolhido (Parte A), em **JSON e CSV**, no mesmo formato usado na aula (campos `codigo_ibge` e `municipio`, com um bloco de proveniência: fonte, endpoint, data da coleta);
- a lista de conjuntos de dados do TSE encontrados na Parte B, em **JSON**, com o termo de busca e o total.

In [189]:
# TODO: grave dados_exercicio/municipios_<uf>.json e .csv (Parte A)
df.to_csv(DADOS / "municipios_rj.csv", index=False, sep=";", decimal=",", encoding="utf8")
df.to_json(DADOS /"municipios.json", index=False)
# TODO: grave dados_exercicio/tse_conjuntos_<termo>.json (Parte B)
df_tse.to_json(DADOS / "temos_tse.json")


# Conferindo o que foi gravado
import os
for arquivo in sorted(DADOS.glob("*")):
    print(arquivo.name, os.path.getsize(arquivo), "bytes")


municipios.json 41257 bytes
municipios_rj.csv 42785 bytes
municipios_uf.json 110646 bytes
temos_tse.json 311154 bytes


---

**Pronto?** Se os três blocos acima rodaram sem erro e os arquivos foram gravados em `dados_exercicio/`, você já praticou tudo que precisa para o **mini-projeto**: requisição GET, paginação, tratamento de erro e exportação em JSON/CSV — só que agora com uma API e um dataset escolhidos por você. Veja o enunciado do mini-projeto no Moodle.